In [1]:
pip install spotipy


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
import spotipy
import requests
from spotipy.oauth2 import SpotifyClientCredentials 
client_id = "d13d846a98144478bc1f7dfe17deca1e"
client_secret = "a243ceec058c4d46b478c895c85dc004"
client_credentials_manager = SpotifyClientCredentials(client_id=client_id, client_secret=client_secret)
sp = spotipy.Spotify(client_credentials_manager=client_credentials_manager) 

In [25]:
name = ["The Chainsmokers"]
result = sp.search(name) 
result['tracks']['items'][1]['artists']

[{'external_urls': {'spotify': 'https://open.spotify.com/artist/69GGBxA162lTqCwzJG5jLp'},
  'href': 'https://api.spotify.com/v1/artists/69GGBxA162lTqCwzJG5jLp',
  'id': '69GGBxA162lTqCwzJG5jLp',
  'name': 'The Chainsmokers',
  'type': 'artist',
  'uri': 'spotify:artist:69GGBxA162lTqCwzJG5jLp'},
 {'external_urls': {'spotify': 'https://open.spotify.com/artist/6jsjhAEteAlY0vCiLvMLBA'},
  'href': 'https://api.spotify.com/v1/artists/6jsjhAEteAlY0vCiLvMLBA',
  'id': '6jsjhAEteAlY0vCiLvMLBA',
  'name': 'ROZES',
  'type': 'artist',
  'uri': 'spotify:artist:6jsjhAEteAlY0vCiLvMLBA'}]

In [31]:
#Extract Artist's uri
artists_uris = result['tracks']['items'][0]['artists'][0]['uri']

token = sp.auth_manager.get_access_token(as_dict=False)
artist_id = artists_uris.split(':')[-1]

resp = requests.get(
    f'https://api.spotify.com/v1/artists/{artist_id}/albums',
    headers={'Authorization': f'Bearer {token}'},
    params={'include_groups': 'album', 'limit': 10}
)
results = resp.json()

#print(results)

artist_album_names = []
artist_album_uris = []
for i in range(len(results['items'])):
    artist_album_names.append(results['items'][i]['name'])
    artist_album_uris.append(results['items'][i]['uri'])
    
artist_album_names
artist_album_uris



['spotify:album:06NJ4sxQJg1BiSGH9WkzRE',
 'spotify:album:3Fpxa7OUrdGmqEebgoAF1E',
 'spotify:album:4nZ4dv1XvDE25Lf2MFhOqA',
 'spotify:album:01GR4NL5O5CZM51k0aejKD',
 'spotify:album:6ZvDJs17O3woQirttKRYCG',
 'spotify:album:4JPguzRps3kuWDD5GS6oXr']

In [32]:
spotify_albums = {}
album_count = 0

def album_songs(uri):
    album = uri 
    spotify_albums[album] = {}
    #Create keys-values of empty lists inside nested dictionary for album
    spotify_albums[album]['album'] = [] 
    spotify_albums[album]['track_number'] = []
    spotify_albums[album]['id'] = []
    spotify_albums[album]['name'] = []
    spotify_albums[album]['uri'] = []
    #pull data on album tracks
    tracks = sp.album_tracks(album) 
    for n in range(len(tracks['items'])): 
        spotify_albums[album]['album'].append(artist_album_names[album_count]) 
        spotify_albums[album]['track_number'].append(tracks['items'][n]['track_number'])
        spotify_albums[album]['id'].append(tracks['items'][n]['id'])
        spotify_albums[album]['name'].append(tracks['items'][n]['name'])
        spotify_albums[album]['uri'].append(tracks['items'][n]['uri'])

for i in artist_album_uris: #each album
    album_songs(i)
    print(str(artist_album_names[album_count]) + " album songs has been added to spotify_albums dictionary")
    album_count+=1 #Updates album count once all tracks have been added

Summertime Friends album songs has been added to spotify_albums dictionary
So Far So Good (lofi remixes) album songs has been added to spotify_albums dictionary
So Far So Good album songs has been added to spotify_albums dictionary
World War Joy album songs has been added to spotify_albums dictionary
Sick Boy album songs has been added to spotify_albums dictionary
Memories...Do Not Open album songs has been added to spotify_albums dictionary


In [35]:
def audio_features(album):
    #Add new key-values to store audio features
    spotify_albums[album]['acousticness'] = []
    spotify_albums[album]['danceability'] = []
    spotify_albums[album]['energy'] = []
    spotify_albums[album]['instrumentalness'] = []
    spotify_albums[album]['liveness'] = []
    spotify_albums[album]['loudness'] = []
    spotify_albums[album]['speechiness'] = []
    spotify_albums[album]['tempo'] = []
    spotify_albums[album]['valence'] = []
    spotify_albums[album]['popularity'] = []
    
    track_count = 0
    for track in spotify_albums[album]['uri']:
        #pull audio features per track
        features = sp.audio_features(track)
        
        #Append to relevant key-value
        spotify_albums[album]['acousticness'].append(features[0]['acousticness'])
        spotify_albums[album]['danceability'].append(features[0]['danceability'])
        spotify_albums[album]['energy'].append(features[0]['energy'])
        spotify_albums[album]['instrumentalness'].append(features[0]['instrumentalness'])
        spotify_albums[album]['liveness'].append(features[0]['liveness'])
        spotify_albums[album]['loudness'].append(features[0]['loudness'])
        spotify_albums[album]['speechiness'].append(features[0]['speechiness'])
        spotify_albums[album]['tempo'].append(features[0]['tempo'])
        spotify_albums[album]['valence'].append(features[0]['valence'])
        #popularity is stored elsewhere
        pop = sp.track(track)
        spotify_albums[album]['popularity'].append(pop['popularity'])
        track_count+=1


In [36]:
import time
import numpy as np
sleep_min = 2
sleep_max = 5
start_time = time.time()
request_count = 0
for i in spotify_albums:
    audio_features(i)
    request_count+=1
    if request_count % 5 == 0:
        print(str(request_count) + " playlists completed")
        time.sleep(np.random.uniform(sleep_min, sleep_max))
        print('Loop #: {}'.format(request_count))
        print('Elapsed Time: {} seconds'.format(time.time() - start_time))

HTTP Error for GET to https://api.spotify.com/v1/audio-features/?ids=2lOY5Ah2pRRMdjA6yBRgwW with Params: {} returned 403 due to None


SpotifyException: http status: 403, code: -1 - https://api.spotify.com/v1/audio-features/?ids=2lOY5Ah2pRRMdjA6yBRgwW:
 None, reason: None

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import plotly.express as px
from plotly.offline import iplot, plot

